# AutoDocs Colab Processor Setup

This notebook sets up the FastAPI processor in Google Colab using **manual upload**.

Before running:
- Runtime -> Change runtime type -> `GPU`
- Add Colab secrets:
  - `HF_TOKEN`
  - `NGROK_AUTHTOKEN`
- When prompted, upload these three files from your repo:
  - `processing-service/app.py`
  - `processing-service/pipeline.py`
  - `processing-service/requirements.txt`


In [ ]:
from google.colab import files

uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))


In [ ]:
!mkdir -p /content/processing-service
!mv /content/app.py /content/processing-service/app.py
!mv /content/pipeline.py /content/processing-service/pipeline.py
!mv /content/requirements.txt /content/processing-service/requirements.txt
%cd /content/processing-service
!pwd
!ls -la


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q pyngrok
!pip install -q -r requirements.txt


In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['MODEL0_REPO_ID'] = 'Jaiccc/model_0_streaming_timestamp'

# Stronger model 1 configuration.
os.environ['MODEL1_MODEL_ID'] = 'openai/gpt-oss-20b'
os.environ['MODEL1_DTYPE'] = 'bfloat16'
os.environ['MODEL1_GPU_UTIL'] = '0.70'
os.environ['MODEL1_MAX_MODEL_LEN'] = '16384'
os.environ['MODEL1_MAX_NEW_TOKENS'] = '2000'

NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')

print('HF token loaded:', bool(os.environ['HF_TOKEN']))
print('ngrok token loaded:', bool(NGROK_AUTHTOKEN))


In [ ]:
get_ipython().system_raw(
    'cd /content/processing-service && '
    'uvicorn app:app --host 0.0.0.0 --port 8000 > /content/processor.log 2>&1 &'
)
print('Started FastAPI service in background')


In [ ]:
import time

time.sleep(8)
!tail -n 100 /content/processor.log


In [ ]:
!curl -s http://127.0.0.1:8000/health


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(8000, 'http')
print('PROCESSOR_API_URL =', tunnel.public_url)


In [ ]:
print(f'''\nPut this in autodocs-app/.env.local:\n\nPROCESSOR_API_URL={tunnel.public_url}\n''')


## Optional direct test

This posts a recording straight to the local FastAPI service before wiring Next.js.


In [ ]:
from google.colab import files
import requests

uploaded = files.upload()
filename = next(iter(uploaded.keys()))

with open(filename, 'rb') as f:
    response = requests.post(
        'http://127.0.0.1:8000/process-terminal-recording',
        files={'file': (filename, f, 'application/octet-stream')},
        data={'title': filename.rsplit('.', 1)[0]},
        timeout=1800,
    )

print('status:', response.status_code)
data = response.json()
print(data.keys())
print('annotations count:', len(data.get('annotations', [])))
print(data.get('sessionContent', '')[:1200])


## Troubleshooting

- If model 1 OOMs, reduce:
  - `MODEL1_MAX_MODEL_LEN` to `8192` or `4096`
  - `MODEL1_GPU_UTIL` to `0.60`
- To inspect logs:

```python
!tail -f /content/processor.log
```
